# Run a Hugging Face model in Colab and keep the weights in Drive

This experiment uses [SmolLM2-135M-Instruct](https://huggingface.co/HuggingFaceTB/SmolLM2-135M-Instruct), a small public chat model. It takes about 273 MB in Drive and runs on a standard Colab CPU. The notebook will use CUDA if you select a GPU runtime.

- **Colab in the browser.** Open [Google Colab](https://colab.research.google.com/), choose **File → Upload notebook**, then connect to a hosted CPU runtime.
- **Colab in VS Code.** Open this file with the [Google Colab extension](https://marketplace.visualstudio.com/items?itemName=Google.colab) and Jupyter installed. Choose **Select Kernel → Colab → Auto Connect** or create a standard CPU server. Sign into the same Google account as in the browser. [Google's extension guide](https://github.com/googlecolab/colab-vscode/wiki/User-Guide#hello-world-)

**First pass.** Run cells 1 through 6 in order. Wait for cell 6 to finish before replacing the runtime.

**Fresh runtime.** In browser Colab, choose **Runtime → Disconnect and delete runtime**, then reconnect. In VS Code, provision a fresh CPU server with **New Colab Server → CPU → Standard**. Run **1, 2, 3, and 5**, skipping the download cell. If you want to try both interfaces, you can also do the first pass in the browser and the fresh-runtime pass in VS Code.

The test passes when cell 5 reports a missing runtime marker and generates a reply from the saved Drive files. The package install in cell 1 may use the internet. Model loading in cell 5 uses local files only.

### 1. Check the Transformers dependency

Colab supplies PyTorch. If its preinstalled Transformers version meets the requirement, pip keeps it.

In [ ]:
%pip install -q "transformers>=4.57.6,<6"

### 2. Mount Google Drive

Run the cell and follow Google's authorization prompt. In VS Code, you can also use **Colab → Mount Google Drive to Server** and run the cell it inserts.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

### 3. Set the model and storage paths

`/content/drive/MyDrive` is the mounted Drive. The test marker lives directly in the runtime's `/content` directory and disappears when the VM is replaced.

In [ ]:
from pathlib import Path

model_id = "HuggingFaceTB/SmolLM2-135M-Instruct"
revision = "12fd25f77366fa6b3b4b768ec3050bf629380bac"
drive_root = Path("/content/drive/MyDrive")
model_dir = drive_root / "colab-models" / model_id.split("/")[-1]
runtime_marker = Path("/content/smollm2-persistence-test.marker")

# File sizes in bytes at the pinned Hugging Face revision.
model_files = {
    "config.json": 861,
    "generation_config.json": 132,
    "merges.txt": 466_391,
    "model.safetensors": 269_060_552,
    "special_tokens_map.json": 655,
    "tokenizer.json": 2_104_556,
    "tokenizer_config.json": 3_764,
    "vocab.json": 800_662,
}

def missing_or_incomplete_files(folder):
    bad_files = []
    for name, expected_size in model_files.items():
        path = folder / name
        if not path.is_file() or path.stat().st_size != expected_size:
            bad_files.append(name)
    return bad_files

if not drive_root.is_dir():
    raise RuntimeError("Mount Google Drive in cell 2 first.")

print(f"Model folder: {model_dir}")

### 4. Download the model once and save it to Drive

Download the weights, tokenizer, and configuration into runtime storage, then copy the missing or incomplete files to Drive. Running this cell again reuses a complete Drive copy.

In [ ]:
from shutil import copyfile
from tempfile import TemporaryDirectory

from huggingface_hub import snapshot_download

files_to_copy = missing_or_incomplete_files(model_dir)
if files_to_copy:
    model_dir.mkdir(parents=True, exist_ok=True)
    download_dir = Path("/content") / f"smollm2-download-{revision[:8]}"
    snapshot_download(
        repo_id=model_id,
        revision=revision,
        local_dir=download_dir,
        allow_patterns=files_to_copy,
        token=False,
    )

    # Publish each file only after its Drive copy is complete.
    with TemporaryDirectory(prefix=".copy-", dir=model_dir) as staging:
        for name in files_to_copy:
            staged_file = Path(staging) / name
            copyfile(download_dir / name, staged_file)
            if staged_file.stat().st_size != model_files[name]:
                raise IOError(f"Incomplete copy: {name}")
            staged_file.replace(model_dir / name)

    print(f"Saved {len(files_to_copy)} file(s) to Drive.")
else:
    print("The complete model is already in Drive.")

runtime_marker.touch()
print(f"Drive copy ready at {model_dir}")

### 5. Generate text using the model stored in Drive

This cell loads the model and tokenizer by their Drive path with `local_files_only=True`. Change `prompt` to try another question.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

incomplete = missing_or_incomplete_files(model_dir)
if incomplete:
    raise FileNotFoundError(f"Drive files missing or incomplete: {', '.join(incomplete)}. Run cell 4 to repair them.")

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
print(f"Loading from {model_dir} on {device}...")

tokenizer = AutoTokenizer.from_pretrained(model_dir, local_files_only=True)
model = AutoModelForCausalLM.from_pretrained(
    model_dir, dtype=dtype, local_files_only=True, use_safetensors=True
).to(device).eval()

prompt = "In one short sentence, what is a solar eclipse?"
inputs = tokenizer.apply_chat_template(
    [{"role": "user", "content": prompt}],
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(device)

with torch.inference_mode():
    output = model.generate(
        **inputs,
        max_new_tokens=48,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
    )

reply = tokenizer.decode(output[0, inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
if not reply.strip():
    raise RuntimeError("Generation returned an empty reply.")

print(f"Model reply: {reply}")
if runtime_marker.exists():
    print("Baseline passed. The runtime marker is still present.")
else:
    print("Persistence check passed. The runtime marker is absent and generation used the Drive copy.")

### 6. Flush Drive before replacing the first runtime

This waits for pending Drive writes and unmounts Drive. Now switch to a fresh runtime and run cells **1, 2, 3, and 5**.

In [ ]:
drive.flush_and_unmount()
print("Drive flush finished. You can now replace the runtime.")